Описание: Краткое введение и назначение ноутбука.
# Bounce-only: подбор множителей и методов уровней

Этот ноутбук содержит только логику и подбор для стратегии отскока (Buy/Sell Limit).
Запускайте ячейки сверху вниз.

In [ ]:
# Описание: импорт библиотек и подавление частых предупреждений
import os
import warnings
from itertools import product
import numpy as np
import pandas as pd
from backtesting import Backtest, Strategy
warnings.filterwarnings(
    "ignore",
    message=".*contingent SL/TP order would execute in the same bar.*",
)

In [ ]:
# Описание: конфигурация эксперимента и сетки параметров для перебора
CONFIG = {
    'data_file': 'EURUSD_H1_2020-01-01_2025-12-31.csv',
    'atr_period': 14,
    'split_date': '2025-01-01',
    'cash': 100_000,
    'leverage': 100,
    'commission': 0.00002,
    'min_trades': 30,
    'top_n': 10,
}
BUFFER_GRID = [0.1, 0.15, 0.2, 0.25]
SL_GRID = [0.5, 1.0, 1.5, 2.0]
TP_GRID = [1.0, 1.5, 2.0, 2.5, 3.0, 4.0]
LEVEL_METHOD_NAMES = {0: 'Camarilla', 1: 'Pivot (классика)', 2: 'DeMark'}
SUP_COLS = {0: ['cam_s1','cam_s2','cam_s3','cam_s4'], 1: ['piv_s1','piv_s2','piv_s3'], 2: ['dem_s1']}
RES_COLS = {0: ['cam_r1','cam_r2','cam_r3','cam_r4'], 1: ['piv_r1','piv_r2','piv_r3'], 2: ['dem_r1']}

In [ ]:
# Описание: функции вычисления дневных уровней (Camarilla, Pivot, DeMark)
def camarilla_levels(d1):
    h,l,c = d1['High'], d1['Low'], d1['Close']
    hl = h-l
    out = pd.DataFrame(index=d1.index)
    for i,div in enumerate([12,6,4,2], start=1):
        out[f'cam_r{i}'] = c + hl * 1.1 / div
        out[f'cam_s{i}'] = c - hl * 1.1 / div
    return out
def pivot_levels(d1):
    h,l,c = d1['High'], d1['Low'], d1['Close']
    p = (h+l+c)/3
    out = pd.DataFrame(index=d1.index)
    out['piv_r1'] = 2*p - l
    out['piv_s1'] = 2*p - h
    out['piv_r2'] = p + (h-l)
    out['piv_s2'] = p - (h-l)
    out['piv_r3'] = h + 2*(p-l)
    out['piv_s3'] = l - 2*(h-p)
    return out
def demark_levels(d1):
    h,l,c,o = d1['High'], d1['Low'], d1['Close'], d1['Open']
    x = pd.Series(np.where(c<o, h+2*l+c, np.where(c>o, 2*h + l + c, h + l + 2*c)), index=d1.index)
    out = pd.DataFrame(index=d1.index)
    out['dem_r1'] = x/2 - l
    out['dem_s1'] = x/2 - h
    return out

In [ ]:
# Описание: загрузка CSV, вычисление дневных уровней и ATR
def load_and_prepare(csv_path, cfg):
    df = pd.read_csv(csv_path)
    df['time'] = pd.to_datetime(df['time'])
    df = df.set_index('time').sort_index()
    df = df.rename(columns={'open':'Open','high':'High','low':'Low','close':'Close','tick_volume':'Volume'})[[
,
,
,
,
]]
    off = pd.Timedelta(hours=0)
    day = (df.index + off).normalize()
    d1 = df.groupby(day).agg(Open=('Open','first'), High=('High','max'), Low=('Low','min'), Close=('Close','last'))
    lev = pd.concat([camarilla_levels(d1), pivot_levels(d1), demark_levels(d1)], axis=1).shift(1)
    lev.index.name = '_day'
    df = df.assign(_day=day).join(lev, on='_day').drop(columns='_day')
    prev = df['Close'].shift(1)
    tr = pd.concat([df['High']-df['Low'], (df['High']-prev).abs(), (df['Low']-prev).abs()], axis=1).max(axis=1)
    df['ATR'] = tr.rolling(cfg['atr_period']).mean()
    return df

In [ ]:
# Описание: класс стратегии отскока — логика открытия лимитных ордеров
class BounceStrategy(Strategy):
    level_method = 0
    buffer_atr_s = 0.5
    buffer_atr_r = 0.5
    sl_atr_s = 2.0
    sl_atr_r = 2.0
    tp_atr_s = 2.0
    tp_atr_r = 2.0
    long_only = False
    short_only = False
    def init(self):
        df = self.data.df
        self._i = 0
        self._close = df['Close'].to_numpy()
        self._atr = df['ATR'].to_numpy()
        self._sup = {c: df[c].to_numpy() for c in SUP_COLS[self.level_method]}
        self._res = {c: df[c].to_numpy() for c in RES_COLS[self.level_method]}
        self._open_positions = 0
    def next(self):
        i = self._i; self._i += 1
        atr = self._atr[i]
        if not np.isfinite(atr) or atr<=0: return
        close = self._close[i]
        if not self.short_only:
            sup = None
            for arr in self._sup.values():
                v = arr[i];
                if v <= close and (sup is None or v>sup): sup = v
            if sup is not None:
                entry = sup + self.buffer_atr_s * atr
                risk = self.sl_atr_s * atr
                size = max(1, int(round((CONFIG['cash']*0.02)/risk)))
                if close>entry: self.buy(size=size, limit=entry, sl=entry-self.sl_atr_s*atr, tp=entry+self.tp_atr_s*atr)
        if not self.long_only:
            res = None
            for arr in self._res.values():
                v = arr[i];
                if v >= close and (res is None or v<res): res = v
            if res is not None:
                entry = res - self.buffer_atr_r * atr
                risk = self.sl_atr_r * atr
                size = max(1, int(round((CONFIG['cash']*0.02)/risk)))
                if close<entry: self.sell(size=size, limit=entry, sl=entry+self.sl_atr_r*atr, tp=entry-self.tp_atr_r*atr)

In [ ]:
# Описание: функции для статистической строки, перебора сеток и форматирования топ-таблицы
def stats_row(s, params):
    return {**params, 'Sharpe': s.get('Sharpe Ratio', np.nan), 'Trades': s.get('# Trades', len(s.get('_trades', []))), 'End Balance': s.get('End Balance', np.nan)}
def grid_search(bt_obj, grids, fixed, cfg):
    keys = list(grids)
    combos = list(product(*grids.values()))
    rows = []
    for combo in combos:
        params = dict(zip(keys, combo))
        params.update({k: v[0] for k,v in fixed.items()})
        s = bt_obj.run(**params)
        rows.append(stats_row(s, params))
    out = pd.DataFrame(rows)
    out = out[out['Trades'] >= cfg['min_trades']]
    if out.empty: out = pd.DataFrame(rows)
    return out.sort_values('Sharpe', ascending=False).reset_index(drop=True)
def top_table(rows, cfg):
    d = rows.head(cfg['top_n']).copy()
    d['Метод'] = d['level_method'].map(LEVEL_METHOD_NAMES)
    cols = ['Метод','buffer_atr_s','sl_atr_s','tp_atr_s','buffer_atr_r','sl_atr_r','tp_atr_r','Sharpe','Trades','End Balance']
    return d[cols]

In [ ]:
# Описание: загрузить файл, отфильтровать по датам и подготовить Backtest объекты
DATA_PATH = next((p for p in ['content/' + CONFIG['data_file'], CONFIG['data_file']] if os.path.exists(p)), None)
if DATA_PATH is None: raise FileNotFoundError(f
)
df = load_and_prepare(DATA_PATH, CONFIG)
df = df.loc['2024':'2025']
split = pd.Timestamp(CONFIG['split_date'])
df_train = df[df.index < split]
df_test = df[df.index >= split]
bt_train = Backtest(df_train, BounceStrategy, cash=CONFIG['cash'], commission=CONFIG['commission'], margin=1/CONFIG['leverage'], finalize_trades=True)
bt_test = Backtest(df_test, BounceStrategy, cash=CONFIG['cash'], commission=CONFIG['commission'], margin=1/CONFIG['leverage'], finalize_trades=True)

In [ ]:
# Проход: ОТСКОК ОТ ПОДДЕРЖКИ — подбор S (Buy Limit)
r1 = grid_search(bt_train, grids={'level_method': range(3), 'buffer_atr_s': BUFFER_GRID, 'sl_atr_s': SL_GRID, 'tp_atr_s': TP_GRID}, fixed={'buffer_atr_r':[0.5],'sl_atr_r':[1.0],'tp_atr_r':[2.0],'long_only':[True],'short_only':[False]}, cfg=CONFIG)
best1 = r1.iloc[0]
display(top_table(r1, CONFIG))

In [ ]:
# Проход: ОТСКОК ОТ СОПРОТИВЛЕНИЯ — подбор R (Sell Limit)
r2 = grid_search(bt_train, grids={'level_method': range(3), 'buffer_atr_r': BUFFER_GRID, 'sl_atr_r': SL_GRID, 'tp_atr_r': TP_GRID}, fixed={'buffer_atr_s':[0.5],'sl_atr_s':[1.0],'tp_atr_s':[2.0],'long_only':[False],'short_only':[True]}, cfg=CONFIG)
best2 = r2.iloc[0]
display(top_table(r2, CONFIG))

In [ ]:
# Проход 3: выбор метода для ЛОНГ и ШОРТ при найденных множителях
r3_long = grid_search(bt_train, grids={'level_method': range(3)}, fixed={'buffer_atr_s':[best1['buffer_atr_s']],'sl_atr_s':[best1['sl_atr_s']],'tp_atr_s':[best1.get('tp_atr_s',2.0)],'buffer_atr_r':[0.5],'sl_atr_r':[1.0],'tp_atr_r':[2.0],'long_only':[True],'short_only':[False]}, cfg=CONFIG)
best_long = r3_long.iloc[0]
display(top_table(r3_long, CONFIG))
r3_short = grid_search(bt_train, grids={'level_method': range(3)}, fixed={'buffer_atr_s':[0.5],'sl_atr_s':[1.0],'tp_atr_s':[2.0],'buffer_atr_r':[best2['buffer_atr_r']],'sl_atr_r':[best2['sl_atr_r']],'tp_atr_r':[best2.get('tp_atr_r',2.0)],'long_only':[False],'short_only':[True]}, cfg=CONFIG)
best_short = r3_short.iloc[0]
display(top_table(r3_short, CONFIG))
# Итоговый вывод
print('--- Лучшие методы (Bounce) ---')
print('ОТСКОК ОТ ПОДДЕРЖКИ :', LEVEL_METHOD_NAMES[int(best_long['level_method'])], 'params S:', {'buffer_atr_s':float(best_long['buffer_atr_s']),'sl_atr_s':float(best_long['sl_atr_s']),'tp_atr_s':float(best_long.get('tp_atr_s',2.0))})
print('ОТСКОК ОТ СОПРОТИВЛЕНИЯ :', LEVEL_METHOD_NAMES[int(best_short['level_method'])], 'params R:', {'buffer_atr_r':float(best_short['buffer_atr_r']),'sl_atr_r':float(best_short['sl_atr_r']),'tp_atr_r':float(best_short.get('tp_atr_r',2.0))})